## Exercise 2 - Rice Coding for Lossless WAV Compression

### Overview

This notebook implements a **lossless audio compression pipeline** using **Rice coding** in Python.  
Audio samples from `.wav` files are encoded into a compact binary `.ex2` format and decoded back,  
with **byte-perfect reconstruction** verified via SHA-256.

### How the Pipeline Works

1. **Install & Import Libraries** - loads all required packages (`wave`, `numpy`, `struct`, `hashlib`, `pandas`)
2. **Read PCM WAV** - reads mono 16-bit audio samples from `.wav` file
3. **Zigzag Mapping** - maps signed integers to nonnegative integers 
   Formula: `(v << 1) ^ (v >> 31)` for encoding; `(u >> 1) ^ -(u & 1)` for decoding
4. **Rice Encoding** - encodes nonnegative values using unary quotient + fixed K-bit remainder  
   Quotient `q = u >> K`, Remainder `r = u & (2^K - 1)`
5. **Serialise to `.ex2`** - packs bitstream + metadata header into a binary file  
   Header: `MAGIC(5s) + sample_rate(I) + channels(H) + sample_width(H) + n_samples(I) + K(B)`
6. **Decode `.ex2` -> WAV** - reverses Rice decode + inverse zigzag to reconstruct original samples
7. **Verify Losslessness** - SHA-256 hash comparison confirms byte-exact reconstruction

### Extension - Delta Prediction (DPCM)

As a custom extension, **first-order delta prediction** is applied before Rice coding:

- **Delta Encode**: replaces each sample with its difference from the previous sample  
  `residual[i] = sample[i] - sample[i-1]`
- **Delta Decode**: recovers original samples via cumulative sum  
  `sample[i] = cumsum(residuals[i])`
- **Why it helps**: residuals cluster near zero -> Rice coding produces shorter codewords -> smaller file

### Key Parameters
- `K = 4` - Rice parameter; `M = 2^4 = 16`; larger K -> more bits for remainder, fewer for unary quotient
- `K = 2` - Rice parameter; `M = 2^2 = 4`; smaller K -> larger unary quotients for same value
- `MAGIC = b"RCEX2"` - 5-byte identifier written to every `.ex2` file header

### Results Summary

| File | K | Without Delta (%) | With Delta (%) |
|---|---|---|---|
| Sound1.wav | 4 | -500.10% | +14.43% |
| Sound1.wav | 2 | -2202.12% | -142.49% |
| Sound2.wav | 4 | -3779.10% | -5499.78% |
| Sound2.wav | 2 | -15319.54% | -22202.25% |

## Install Necessary Packages & Libraries

In [1]:
%pip install tabulate pandas numpy

# Import Library
import csv
# For SHA-256 lossless verification
import hashlib     
import struct       
import wave         
import os          
import numpy as np  
import pandas as pd 
from dataclasses import dataclass       
from pathlib import Path                
from typing import List, Sequence

# Magic bytes used to identify our custom .ex2 file format
MAGIC = b"RCEX2"

Note: you may need to restart the kernel to use updated packages.


## WavPayload - Audio Data Container

A `dataclass` that holds all metadata and samples needed to reconstruct a WAV file after decoding.

`Fields:`
- sample_rate : Number of samples per second
- channels : Number of audio channels
- sample_width : Bytes per sample
- samples : NumPy array of int16 audio sample values

In [2]:
@dataclass
class WavPayload:
    sample_rate: int
    channels: int
    sample_width: int
    samples: np.ndarray

## Read WAV File

Reads a WAV file and returns its audio as **mono 16-bit PCM** samples.

### Steps
1. **Open file** - loads WAV in read-binary mode
2. **Extract metadata** - sample rate, channels, sample width, frame count
3. **Read frames** - all audio frames read as raw bytes
4. **Convert to int16** - bytes converted to a NumPy `int16` array
5. **Downmix to mono** - if stereo, only the first (left) channel is kept

In [3]:
def read_wav_mono(path: Path) -> WavPayload:
    with wave.open(str(path), "rb") as wf:
        channels = wf.getnchannels()      
        sample_width = wf.getsampwidth()  
        sample_rate = wf.getframerate()   
        n_frames = wf.getnframes()        
        raw = wf.readframes(n_frames)     

    # Only 16-bit PCM is supported
    if sample_width != 2:
        raise ValueError("Only 16-bit PCM WAV is supported.")

    # Convert raw bytes to signed 16-bit integer array
    samples = np.frombuffer(raw, dtype=np.int16)

    # If stereo, reshape to (n_frames, 2) and keep only first channel
    if channels > 1:
        samples = samples.reshape(-1, channels)[:, 0]

    return WavPayload(
        sample_rate=sample_rate,
        channels=1,
        sample_width=sample_width,
        samples=samples.copy(),
    )

## Write WAV File

Writes a `WavPayload` object back to a `.wav` file on disk.

### Steps
1. **Create directories** - parent folders are created if they do not exist
2. **Open file** - WAV opened in write-binary mode
3. **Set parameters** - channels, sample width, and sample rate applied
4. **Write frames** - `int16` samples written as raw bytes

In [4]:
def write_wav_mono(path: Path, payload: WavPayload) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(payload.sample_width)
        wf.setframerate(payload.sample_rate)
        wf.writeframes(payload.samples.astype(np.int16).tobytes())

## Zigzag Mapping - Signed <-> Unsigned

Rice coding requires **nonnegative integers**. Since audio residuals are signed (`int16`), they must be mapped first.

### Mapping
- `signed_to_unsigned` - maps signed -> nonneg: `0->0`, `-1->1`, `1->2`, `-2->3`, `2->4` ...
- `unsigned_to_signed` - exact inverse mapping back to original signed value

In [5]:
def signed_to_unsigned(arr: np.ndarray) -> np.ndarray:
    a = arr.astype(np.int32)
    return np.where(a >= 0, a << 1, ((-a) << 1) - 1).astype(np.uint32)

def unsigned_to_signed(u: np.ndarray) -> np.ndarray:
    u = u.astype(np.uint32)
    return np.where(u % 2 == 0, u // 2, -((u + 1) // 2)).astype(np.int32)

## Rice Encoding

Rice coding is a lossless compression method that works well when most values
are small and close to zero. I implemented it by splitting each value into a
**quotient** and a **remainder** based on the Rice parameter `K`.

### Output
`[4 bytes: total bit count] + [packed bitstream]`

In [6]:
def rice_encode(samples: np.ndarray, k: int) -> bytes:
    arr = samples.astype(np.int32)
    u   = signed_to_unsigned(arr)

    q = (u >> k).astype(np.uint32)
    r = (u & ((1 << k) - 1)).astype(np.uint32)

    # Codeword start positions in the bitstream
    codeword_lengths = (q + 1 + k).astype(np.int64)
    starts           = np.zeros(len(u) + 1, dtype=np.int64)
    np.cumsum(codeword_lengths, out=starts[1:])
    total_bits = int(starts[-1])

    # Ffill unary ones using cumsum marker trick
    # Set +1 at codeword start, -1 at delimiter position -> cumsum gives 1s in between
    delim_pos = (starts[:-1] + q).astype(np.int64)
    marker    = np.zeros(total_bits + 1, dtype=np.int32)
    np.add.at(marker, starts[:-1], 1)
    np.add.at(marker, delim_pos,  -1)
    bitstream          = (np.cumsum(marker[:-1]) > 0).astype(np.uint8)
    bitstream[delim_pos] = 0  # delimiter zeros

    # Place remainder bits using vectorised index array
    if k > 0:
        rem_starts  = delim_pos + 1
        rem_indices = (rem_starts[:, None] + np.arange(k)).astype(np.int64)
        shifts      = np.arange(k - 1, -1, -1, dtype=np.uint32)
        rem_bits    = ((r[:, None] >> shifts) & 1).astype(np.uint8)
        bitstream[rem_indices] = rem_bits

    # Pack bits to bytes using np.packbits
    pad    = (8 - total_bits % 8) % 8
    padded = np.concatenate([bitstream, np.zeros(pad, dtype=np.uint8)])
    packed = np.packbits(padded)

    return struct.pack("<I", total_bits) + packed.tobytes()

## Rice Decoding

The decoder reverses everything done in `rice_encode()` to perfectly
reconstruct the original audio samples from the binary `.ex2` file.

### Output
A signed `int16` array that is byte-identical to the original WAV samples,
confirmed by SHA-256 hash verification.

In [7]:
def rice_decode(encoded: bytes, k: int, expected_count: int) -> np.ndarray:
    total_bits = struct.unpack("<I", encoded[:4])[0]
    raw        = np.frombuffer(encoded[4:], dtype=np.uint8)

    # Unpack all bits at once
    bits     = np.unpackbits(raw)[:total_bits]
    zero_pos = np.where(bits == 0)[0]  # all delimiter positions

    codeword_starts = np.empty(expected_count, dtype=np.int64)
    delimiters      = np.empty(expected_count, dtype=np.int64)

    # Walk codewords using searchsorted
    pos = 0
    for i in range(expected_count):
        idx = int(np.searchsorted(zero_pos, pos))
        if idx >= len(zero_pos):
            break
        delimiters[i]      = zero_pos[idx]
        codeword_starts[i] = pos
        pos                = int(zero_pos[idx]) + k + 1

    q = (delimiters - codeword_starts).astype(np.uint32)

    if k > 0:
        rem_starts  = delimiters + 1
        rem_indices = (rem_starts[:, None] + np.arange(k)).astype(np.int64)
        shifts      = np.arange(k - 1, -1, -1, dtype=np.uint32)
        rem_bits    = bits[rem_indices]
        r           = (rem_bits * (1 << shifts)).sum(axis=1).astype(np.uint32)
    else:
        r = np.zeros(expected_count, dtype=np.uint32)

    u = ((q << k) | r).astype(np.uint32)
    return unsigned_to_signed(u).astype(np.int16)

## SHA-256 Lossless Verification

Computes the **SHA-256 hash** of a file to verify byte-exact reconstruction.  
If the hash of the decoded WAV matches the original, compression is confirmed **perfectly lossless**.  
File is read in 1 MB chunks to handle large audio files efficiently.

In [8]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

In [9]:
def encode_wav_to_ex2(wav_path: Path, ex2_path: Path, k: int) -> None:
    payload = read_wav_mono(wav_path)
    body    = rice_encode(payload.samples, k)  # raw samples — no delta
    header  = struct.pack(
        "<5sIHHIB", MAGIC,
        payload.sample_rate, payload.channels,
        payload.sample_width, len(payload.samples), k
    )
    Path(ex2_path).parent.mkdir(parents=True, exist_ok=True)
    Path(ex2_path).write_bytes(header + body)


def decode_ex2_to_wav(ex2_path: Path, wav_out_path: Path) -> None:
    blob  = Path(ex2_path).read_bytes()
    hfmt  = "<5sIHHIB"
    hlen  = struct.calcsize(hfmt)
    magic, sr, ch, sw, n, k = struct.unpack(hfmt, blob[:hlen])
    if magic != MAGIC:
        raise ValueError("Not a valid EX2 file")
    samples = rice_decode(blob[hlen:], k, n)
    write_wav_mono(wav_out_path, WavPayload(sr, ch, sw, samples))

## Results Without Delta Prediction

Running Rice coding directly on raw PCM samples without any prediction.

In [10]:
base_dir   = Path(".").resolve()
input_dir  = base_dir / "Exercise2_Files"
output_dir = base_dir / "outputs_no_delta"
output_dir.mkdir(parents=True, exist_ok=True)

rows_no_delta = []

for wav_name in ["Sound1.wav", "Sound2.wav"]:
    src           = input_dir / wav_name
    original_size = src.stat().st_size
    row           = {"File": wav_name, "Original Size": original_size}

    for k in [4, 2]:
        enc = output_dir / f"{src.stem}_NoDelta_K{k}.ex2"
        dec = output_dir / f"{src.stem}_NoDelta_K{k}_dec.wav"

        print(f"Encoding {wav_name} K={k}...\n", end=" ", flush=True)
        encode_wav_to_ex2(src, enc, k)

        decode_ex2_to_wav(enc, dec)

        ok  = sha256(src) == sha256(dec)
        cs  = enc.stat().st_size
        pct = 100.0 * (1.0 - cs / original_size)

        row[f"Rice K={k} Size"] = cs
        row[f"% Compression K={k}"] = f"{pct:.2f}%"

    rows_no_delta.append(row)

df_no_delta = pd.DataFrame(rows_no_delta)
print("\nTable 2.1 - Without Delta Prediction")
display(df_no_delta)

Encoding Sound1.wav K=4...
Encoding Sound1.wav K=2...
Encoding Sound2.wav K=4...
Encoding Sound2.wav K=2...
 
Table 2.1 - Without Delta Prediction


,File,Original Size,Rice K=4 Size,% Compression K=4,Rice K=2 Size,% Compression K=2
0,Sound1.wav,1002088,6013569,-500.10%,23069260,-2202.12%
1,Sound2.wav,1008044,39102991,-3779.10%,155435731,-15319.54%


## Delta Encoding (DPCM) - Extension

Instead of encoding raw audio samples (range ±32767), we encode the **difference between consecutive samples**.  
Differences are typically small, concentrating values near zero - exactly what Rice coding compresses efficiently.

### Wrap-Around Arithmetic
Residuals are kept in `[-32768, 32767]` using modulo 65536 to handle jumps across the `int16` boundary safely.

- `delta_encode` - computes residuals. first sample stored as-is
- `delta_decode` - reconstructs original samples by cumulative sum with the same wrap-around

In [11]:
def delta_encode(samples: np.ndarray) -> np.ndarray:
    s       = samples.astype(np.int32)
    out     = np.empty_like(s)
    out[0]  = int(s[0])
    diff    = s[1:] - s[:-1]
    out[1:] = ((diff + 32768) % 65536) - 32768
    return out


def delta_decode(delta: np.ndarray) -> np.ndarray:
    values  = np.array(delta, dtype=np.int32)
    decoded = np.cumsum(values).astype(np.int32)
    decoded = ((decoded + 32768) % 65536) - 32768
    return decoded.astype(np.int16)

In [12]:
def encode_wav_to_ex2_delta(wav_path: Path, ex2_path: Path, k: int) -> None:
    payload   = read_wav_mono(wav_path)
    residuals = delta_encode(payload.samples)   # Extension: delta first
    body      = rice_encode(residuals, k)
    header    = struct.pack(
        "<5sIHHIB", MAGIC,
        payload.sample_rate, payload.channels,
        payload.sample_width, len(payload.samples), k
    )
    Path(ex2_path).parent.mkdir(parents=True, exist_ok=True)
    Path(ex2_path).write_bytes(header + body)


def decode_ex2_to_wav_delta(ex2_path: Path, wav_out_path: Path) -> None:
    blob  = Path(ex2_path).read_bytes()
    hfmt  = "<5sIHHIB"
    hlen  = struct.calcsize(hfmt)
    magic, sr, ch, sw, n, k = struct.unpack(hfmt, blob[:hlen])
    if magic != MAGIC:
        raise ValueError("Not a valid EX2 file")
    residuals = rice_decode(blob[hlen:], k, n)
    samples   = delta_decode(residuals)
    write_wav_mono(wav_out_path, WavPayload(sr, ch, sw, samples))

## Results With Delta Prediction

Running Rice coding on delta residuals - custom extension beyond specification.

In [13]:
output_dir_delta = base_dir / "outputs_with_delta"
output_dir_delta.mkdir(parents=True, exist_ok=True)

rows_delta = []

for wav_name in ["Sound1.wav", "Sound2.wav"]:
    src           = input_dir / wav_name
    original_size = src.stat().st_size
    row           = {"File": wav_name, "Original Size": original_size}

    for k in [4, 2]:
        enc = output_dir_delta / f"{src.stem}_Delta_K{k}.ex2"
        dec = output_dir_delta / f"{src.stem}_Delta_K{k}_dec.wav"

        print(f"Encoding {wav_name} K={k}...\n", end=" ", flush=True)
        encode_wav_to_ex2_delta(src, enc, k)

        decode_ex2_to_wav_delta(enc, dec)

        ok  = sha256(src) == sha256(dec)
        cs  = enc.stat().st_size
        pct = 100.0 * (1.0 - cs / original_size)

        row[f"Rice K={k} Size"] = cs
        row[f"% Compression K={k}"] = f"{pct:.2f}%"

    rows_delta.append(row)

df_delta = pd.DataFrame(rows_delta)
print("\nTable 2.2 — With Delta Prediction (Extension)")
display(df_delta)

Encoding Sound1.wav K=4...
Encoding Sound1.wav K=2...
Encoding Sound2.wav K=4...
Encoding Sound2.wav K=2...
 
Table 2.2 — With Delta Prediction (Extension)


,File,Original Size,Rice K=4 Size,% Compression K=4,Rice K=2 Size,% Compression K=2
0,Sound1.wav,1002088,857473,14.43%,2429927,-142.49%
1,Sound2.wav,1008044,56448285,-5499.78%,224816526,-22202.25%


## Comparison
Impact of delta prediction extension on compression performance.

In [14]:
comparison = []
files = ["Sound1.wav", "Sound2.wav"]
for i, wav_name in enumerate(files):
    for k in [4, 2]:
        no_delta_pct = rows_no_delta[i][f"% Compression K={k}"]
        delta_pct    = rows_delta[i][f"% Compression K={k}"]
        comparison.append({
            "File"             : wav_name,
            "K"                : k,
            "Without Delta (%)" : no_delta_pct,
            "With Delta (%)"    : delta_pct,
        })

df_comparison = pd.DataFrame(comparison)
print("\nTable 2.3 — Without vs With Delta Prediction")
display(df_comparison)


Table 2.3 — Without vs With Delta Prediction


,File,K,Without Delta (%),With Delta (%)
0,Sound1.wav,4,-500.10%,14.43%
1,Sound1.wav,2,-2202.12%,-142.49%
2,Sound2.wav,4,-3779.10%,-5499.78%
3,Sound2.wav,2,-15319.54%,-22202.25%
